In [ ]:
# ======================================================
# RF + IBCO-optimised BiLSTM residual corrector
# VERSION 7 - Proposed Improved Bitterling Colony Optimization (IBCO)
#
# Implements the paper's IBCO (Eqs. 9-51):
#   - Mussel position Bn = Xbest + H*c*Tempg      (Eqs. 9-16)
#   - Dynamic step L                               (Eq. 17)
#   - Sine-Cosine guided travel behaviour (TB)     (Eqs. 18-28)
#   - DPCM eviction behaviour (EB)                 (Eqs. 29-31)
#   - Reproductive behaviour (RB) male / female    (Eqs. 26, 32)
#   - Adaptive dispersal probability               (Eq. 33)
#   - MSE fitness                                  (Eqs. 34-35)
#   - Greedy acceptance                            (Eq. 36)
#   - Colony transition probabilities              (Eqs. 37-40)
#   - Diversity measure + scattering strategy      (Eqs. 41-45)
#   - Exploration/exploitation ratios              (Eqs. 46-48)
#   - Convergence criterion                        (Eq. 49)
#
# Total tuned parameters = 14 (RF) + 9 (BiLSTM) = 23  (as in the abstract)
# Other paper alignments:
#   - split: 75% train (10% of it = validation) / 25% test
#   - residual + derivative min-max normalised to [0, 1]
#   - both tuners use validation MSE as fitness
#   - extra metrics: MASE, PICP, PINAW, AWD, ACE
# ======================================================
import os, gc, json, warnings, traceback
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras import backend as K
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt


SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        strategy = tf.distribute.OneDeviceStrategy("/gpu:0")
    except RuntimeError:
        strategy = tf.distribute.get_strategy()
else:
    strategy = tf.distribute.get_strategy()

# ======================================================
# Part-I Data + split 
# ======================================================
FILE_PATH = ("/mnt/f/Solar-Review-Ppaer/Monthwise-Data-Sheets/"
             "Data_clean/10-Minute-Average-Data/10-min-AVG-Data.xlsx")
raw = pd.read_excel(FILE_PATH)[['DateTime', 'Power_MW']]
raw['Power_MW'] = raw['Power_MW'].fillna(0)
raw['DateTime'] = pd.to_datetime(raw['DateTime'])
raw.set_index('DateTime', inplace=True)

N       = len(raw)
n_trval = int(N * 0.75)
n_val   = int(n_trval * 0.10)
n_tr    = n_trval - n_val
raw_train = raw.iloc[:n_tr].copy()
raw_val   = raw.iloc[n_tr:n_trval].copy()
raw_test  = raw.iloc[n_trval:].copy()
print(f"Train={len(raw_train)}  Val={len(raw_val)}  Test={len(raw_test)}")

# ======================================================
# Part-II Feature
# ======================================================
def add_cyclic_features(df):
    df = df.copy()
    df['hour'], df['minute'] = df.index.hour, df.index.minute
    df['dayofweek'], df['month'] = df.index.dayofweek, df.index.month
    for c, M in [('hour', 24), ('minute', 60), ('dayofweek', 7)]:
        df[f'{c}_sin'] = np.sin(2 * np.pi * df[c] / M)
        df[f'{c}_cos'] = np.cos(2 * np.pi * df[c] / M)
    df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 12)
    return df

def decompose_train(s, period=144):
    da = seasonal_decompose(s, model='additive', period=period)
    dm = seasonal_decompose(s + 1e-3, model='multiplicative', period=period)
    out = pd.DataFrame({'trend_add': da.trend, 'seasonal_add': da.seasonal,
                        'resid_add': da.resid, 'trend_mul': dm.trend,
                        'seasonal_mul': dm.seasonal, 'resid_mul': dm.resid},
                       index=s.index)
    return out.bfill().ffill()

def apply_decomp_to_split(df, td, period=144):
    n = len(df)
    r = pd.DataFrame(index=df.index)
    r['trend_add'] = float(td['trend_add'].iloc[-1])
    r['trend_mul'] = float(td['trend_mul'].iloc[-1])
    r['seasonal_add'] = np.tile(td['seasonal_add'].values[-period:], int(np.ceil(n / period)))[:n]
    r['seasonal_mul'] = np.tile(td['seasonal_mul'].values[-period:], int(np.ceil(n / period)))[:n]
    r['resid_add'] = df['Power_MW'].values - r['trend_add'].values - r['seasonal_add'].values
    r['resid_mul'] = df['Power_MW'].values / (r['trend_mul'].values * r['seasonal_mul'].values + 1e-8)
    return r

CYCLIC_COLS = ['hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
               'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos']
DECOMP_COLS = ['trend_add', 'seasonal_add', 'resid_add',
               'trend_mul', 'seasonal_mul', 'resid_mul']

tr_c, va_c, te_c = (add_cyclic_features(d) for d in (raw_train, raw_val, raw_test))
tr_d = decompose_train(raw_train['Power_MW'])
va_d = apply_decomp_to_split(raw_val, tr_d)
te_d = apply_decomp_to_split(raw_test, tr_d)
build = lambda c, d: pd.concat([c[CYCLIC_COLS], d[DECOMP_COLS]], axis=1).values

feat_scaler = StandardScaler()
X_train = feat_scaler.fit_transform(build(tr_c, tr_d))
X_val   = feat_scaler.transform(build(va_c, va_d))
X_test  = feat_scaler.transform(build(te_c, te_d))
y_train, y_val, y_test = (d['Power_MW'].values for d in (raw_train, raw_val, raw_test))

# ======================================================
# Part-III Improved Bitterling Colony Optimization (IBCO)
# ======================================================
class IBCO:
    """
    Generic IBCO minimiser.
    fitness_fn(position) -> (fitness: float, aux: any)   aux = e.g. trained model
    """
    def __init__(self, fitness_fn, low, up, pop_size=30, max_iter=200,
                 a=2.0, r1_min=0.05,          # SCA coefficients (Eq. 18)
                 p_dis_min=0.2, p_dis_max=0.9,  # Eq. 33
                 p_evict=0.2,                 # prob. of EB inside the dispersal branch
                 net=10,                      # DPCM grid cells per dimension (Eq. 29)
                 theta=0.05,                  # diversity threshold (Eq. 44)
                 scatter_frac=0.30,           # worst fraction re-scattered
                 conv_p=20, conv_eps=1e-4,    # Eq. 49
                 eval_female=False, name="IBCO", checkpoint=None):
        self.f, self.low, self.up = fitness_fn, np.asarray(low, float), np.asarray(up, float)
        self.dim, self.n, self.T = len(low), pop_size, max_iter
        self.a, self.r1_min = a, r1_min
        self.pmin, self.pmax, self.p_evict = p_dis_min, p_dis_max, p_evict
        self.net, self.theta, self.scatter_frac = net, theta, scatter_frac
        self.conv_p, self.conv_eps = conv_p, conv_eps
        self.eval_female, self.name, self.checkpoint = eval_female, name, checkpoint
        self.best_x, self.best_f, self.best_aux = None, np.inf, None
        self.history, self.div_history = [], []

    # ---- helpers --------------------------------------------------
    def _clip(self, x):
        return np.clip(x, self.low, self.up)

    def _diversity(self, pop):                              # Eqs. 41-43
        xm = pop.mean(axis=0)
        div = np.mean(np.linalg.norm(pop - xm, axis=1))
        return div / np.linalg.norm(self.up - self.low)

    def _mussel(self, it):                                  # Eqs. 9-16
        H = 2 * ((self.T - it + 1) / self.T) ** 2 * np.sin(2 * np.pi * np.random.rand())   # Eq. 10
        k = int(np.ceil(np.random.rand() * self.dim)) if it < 0.25 * self.T else 1       # Eq. 13
        k = max(1, min(k, self.dim))
        g = np.random.permutation(self.dim)[:k]             # Eq. 12
        c = np.zeros(self.dim); c[g] = 1.0                  # Eq. 11
        temp = np.round(0.5 * (0.5 + np.random.rand(self.dim)))    # Eqs. 15-16
        tempg = (1 - temp) * np.abs(self.up - self.best_x) + temp * np.abs(self.low - self.best_x)  # Eq. 14
        return self._clip(self.best_x + H * c * tempg)      # Eq. 9

    def _step_L(self, it):                                  # Eq. 17
        return 2 * ((self.T - it + 1) / self.T) ** 0.5 * np.sin(2 * np.pi * np.random.rand())

    def _sca_travel(self, x, it):                           # Eqs. 18-25
        r1 = max(self.a - self.a * it / self.T, self.r1_min)
        r2 = np.random.uniform(0, 2 * np.pi)
        r3 = np.random.uniform(0, 2)
        r4 = np.random.rand()
        dest = np.abs(r3 * self.best_x - x)                 # Eq. 22
        return x + r1 * (np.sin(r2) if r4 < 0.5 else np.cos(r2)) * dest

    def _dpcm_evict(self, pop):                             # Eqs. 29-31
        lnet = (self.up - self.low) / self.net
        new = np.empty(self.dim)
        for d in range(self.dim):
            cell = np.clip(((pop[:, d] - self.low[d]) / max(lnet[d], 1e-12)).astype(int), 0, self.net - 1)
            crowd = np.bincount(cell, minlength=self.net)
            idx = int(np.argmin(crowd))                     # least crowded cell (Eq. 30)
            new[d] = self.low[d] + (idx + np.random.rand()) * lnet[d]   # sampled inside that cell
        return new

    def _p_dis(self, it):                                   # Eq. 33
        return self.pmin + (self.pmax - self.pmin) * (1 - it / self.T)

    def _converged(self):                                   # Eq. 49
        h, p = self.history, self.conv_p
        if len(h) <= p:
            return False
        return abs(h[-1] - h[-1 - p]) / (abs(h[-1 - p]) + 1e-12) < self.conv_eps

    # ---- main loop ------------------------------------------------
    def optimize(self):
        print("\n" + "=" * 70)
        print(f"IBCO - {self.name}   dim={self.dim}  pop={self.n}  iters={self.T}")
        print("=" * 70)
        pop = self.low + np.random.rand(self.n, self.dim) * (self.up - self.low)
        fit = np.empty(self.n); aux = [None] * self.n
        for i in range(self.n):
            fit[i], aux[i] = self.f(pop[i])
        b = int(np.argmin(fit))
        self.best_x, self.best_f, self.best_aux = pop[b].copy(), float(fit[b]), aux[b]
        self.history.append(self.best_f)
        print(f"  init best fitness (MSE) = {self.best_f:.6f}")

        for it in range(1, self.T + 1):
            Pdis = self._p_dis(it)
            L = self._step_L(it)
            inv = 1.0 / (fit + 1e-12)                       # Eq. 37
            Pbb = inv.max() / inv.sum()                     # Eq. 38
            Pib = inv / inv.sum()                           # Eq. 39
            Bn = self._mussel(it)

            for i in range(self.n):
                xm = pop[i]
                if np.random.rand() < min(1.0, Pib[i] * self.n) * 0.2:      # colony transition (Eq. 40)
                    cand = xm + np.random.rand() * (self.best_x - xm) * Pbb * self.n
                elif np.random.rand() >= Pdis:                              # SCA travel (Eq. 27)
                    cand = self._sca_travel(xm, it)
                elif np.random.rand() < self.p_evict:                       # EB (Eq. 31)
                    cand = self._dpcm_evict(pop)
                else:                                                       # RB (Eq. 26)
                    cand = Bn + L * (xm - Bn)
                cand = self._clip(cand)

                cands = [cand]
                if self.eval_female:                                        # Eq. 32
                    cands.append(self._clip(cand + L * (pop[i] - cand)))

                for c in cands:                                             # greedy acceptance (Eq. 36)
                    fc, ac = self.f(c)
                    if fc < fit[i]:
                        pop[i], fit[i], aux[i] = c, fc, ac
                        if fc < self.best_f:
                            self.best_x, self.best_f, self.best_aux = c.copy(), fc, ac

            div = self._diversity(pop)
            self.div_history.append(div)
            if div < self.theta:                                            # scattering (Eqs. 44-45)
                k = max(1, int(self.scatter_frac * self.n))
                for w in np.argsort(fit)[-k:]:
                    pop[w] = self.low + np.random.rand(self.dim) * (self.up - self.low)
                    fit[w], aux[w] = self.f(pop[w])
                    if fit[w] < self.best_f:
                        self.best_x, self.best_f, self.best_aux = pop[w].copy(), float(fit[w]), aux[w]
                print(f"  [it {it}] diversity {div:.3f} < theta -> scattered {k}")

            self.history.append(self.best_f)
            XPL = div / max(max(self.div_history), 1e-12)                   # Eq. 46
            XPT = 1 - XPL                                                   # Eqs. 47-48
            print(f"  [it {it:3d}/{self.T}] best={self.best_f:.6f}  div={div:.3f}  "
                  f"XPL={XPL:.2f} XPT={XPT:.2f}  Pdis={Pdis:.2f}  L={L:+.2f}")

            if self.checkpoint:
                with open(self.checkpoint, 'w') as fh:
                    json.dump({'it': it, 'best_f': self.best_f,
                               'best_x': self.best_x.tolist(), 'history': self.history}, fh)
            if self._converged():
                print(f"  converged (Eq. 49) at iteration {it}")
                break
        return self.best_x, self.best_f, self.best_aux

# ======================================================
# Part-IV RF search space (14 parameters)
# ======================================================
RF_LOW = np.array([10, 0, 1, 2, 1, 0.0, 0, 10, 0.0, 0, 0, 0, 0.0, 0.1], float)
RF_UP  = np.array([1000, 1, 100, 50, 20, 0.5, 1, 1000, 0.1, 1, 1, 1, 0.1, 1.0], float)

def decode_rf(p):
    boot = bool(round(np.clip(p[9], 0, 1)))
    depth = int(np.clip(round(p[2]), 1, 100))
    leaf = int(np.clip(round(p[7]), 10, 1000))
    ms = float(np.clip(p[13], 0.1, 1.0)) if boot else None
    if ms is not None and ms >= 1.0:
        ms = None
    return dict(
        n_estimators=int(np.clip(round(p[0]), 10, 1000)),
        criterion=['squared_error', 'absolute_error'][int(np.clip(round(p[1]), 0, 1))],
        max_depth=None if depth >= 100 else depth,
        min_samples_split=int(np.clip(round(p[3]), 2, 50)),
        min_samples_leaf=int(np.clip(round(p[4]), 1, 20)),
        min_weight_fraction_leaf=float(np.clip(p[5], 0, 0.5)),
        max_features=[None, 'sqrt'][int(np.clip(round(p[6]), 0, 1))],
        max_leaf_nodes=None if leaf >= 1000 else leaf,
        min_impurity_decrease=float(np.clip(p[8], 0, 0.1)),
        bootstrap=boot,
        oob_score=bool(round(np.clip(p[10], 0, 1))) if boot else False,
        warm_start=bool(round(np.clip(p[11], 0, 1))),
        ccp_alpha=float(np.clip(p[12], 0, 0.1)),
        max_samples=ms)

_rng = np.random.default_rng(SEED)
RF_SUB = _rng.choice(len(X_train), min(6000, len(X_train)), replace=False)

def rf_fitness(pos):
    try:
        rf = RandomForestRegressor(**decode_rf(pos), random_state=SEED, n_jobs=-1)
        rf.fit(X_train[RF_SUB], y_train[RF_SUB])
        return float(mean_squared_error(y_val, rf.predict(X_val))), None   # Eq. 35
    except Exception:
        traceback.print_exc()
        return 1e6, None

# ======================================================
# Part-V BiLSTM search space (9 parameters)
# ======================================================
BCO_SEQ_LEN, FINAL_SEQ_LEN, LSTM_SUBSET = 36, 72, 6000
BI_LOW = np.array([1, 16, 0.1, 0.0, -4.0, 0, 0, 16, 10], float)
BI_UP  = np.array([4, 256, 0.6, 0.4, -2.0, 1, 1, 256, 150], float)

def decode_bilstm(p):
    return {'n_layers': int(np.clip(round(p[0]), 1, 4)),
            'units': int(np.clip(round(p[1]), 16, 256)),
            'dropout': float(np.clip(p[2], 0.1, 0.6)),
            'recurrent_dropout': float(np.clip(p[3], 0.0, 0.4)),
            'log_lr': float(np.clip(p[4], -4, -2)),
            'optimizer': ['adam', 'rmsprop'][int(np.clip(round(p[5]), 0, 1))],
            'activation': ['relu', 'tanh'][int(np.clip(round(p[6]), 0, 1))],
            'batch_size': int(np.clip(round(p[7]), 16, 256)),
            'epochs': int(np.clip(round(p[8]), 10, 150))}

def make_bilstm(params, input_shape, use_recurrent_dropout=True):
    rd = params['recurrent_dropout'] if use_recurrent_dropout else 0.0
    with strategy.scope():
        m = Sequential([tf.keras.layers.Input(shape=input_shape)])
        for i in range(params['n_layers']):
            m.add(Bidirectional(LSTM(params['units'],
                                     return_sequences=i < params['n_layers'] - 1,
                                     activation=params['activation'],
                                     recurrent_dropout=rd)))
            m.add(Dropout(params['dropout']))
        m.add(Dense(1))
        opt = {'adam': Adam, 'rmsprop': RMSprop}[params['optimizer']](
            learning_rate=10 ** params['log_lr'], clipnorm=1.0)
        m.compile(optimizer=opt, loss='mse', metrics=['mae'])
    return m

def make_sequences(data, L):
    Xs = np.stack([data[i:i + L] for i in range(len(data) - L)])
    ys = data[L:, 0]
    return Xs, ys

# ======================================================
# 6. Metrics
# ======================================================
def mase(y, yp, y_train_ref):
    naive = np.mean(np.abs(np.diff(y_train_ref)))
    return float(np.mean(np.abs(y - yp)) / max(naive, 1e-12))

def interval_metrics(y, yp, resid_ref, level):
    """Residual-quantile prediction intervals -> PICP, PINAW, AWD, ACE."""
    a = (1 - level) / 2
    lo_q, hi_q = np.quantile(resid_ref, [a, 1 - a])
    lo, hi = yp + lo_q, yp + hi_q
    inside = (y >= lo) & (y <= hi)
    picp = inside.mean()
    width = np.maximum(hi - lo, 1e-12)
    pinaw = np.mean(hi - lo) / (y.max() - y.min() + 1e-12)
    dist = np.where(y < lo, lo - y, np.where(y > hi, y - hi, 0.0))
    awd = np.mean(dist / width)
    ace = abs(picp - level)
    return dict(level=level, PICP=100 * picp, PINAW=pinaw, AWD=awd, ACE=100 * ace)

# ======================================================
# Part-VI Run-Main
# ======================================================
if __name__ == "__main__":

    # ---- Stage 1: IBCO -> Random Forest ----
    rf_opt = IBCO(rf_fitness, RF_LOW, RF_UP, pop_size=30, max_iter=200,
                  eval_female=True, name="Random Forest (14 params)",
                  checkpoint='rf_ibco_checkpoint.json')
    rf_x, rf_f, _ = rf_opt.optimize()
    rf_params = decode_rf(rf_x)
    print("\nBest RF hyper-parameters:")
    for k, v in rf_params.items():
        print(f"  {k:>26}: {v}")

    # Final RF: fit on TRAIN only so train residuals are honest (no val leakage)
    rf_final = RandomForestRegressor(**rf_params, random_state=SEED, n_jobs=-1)
    rf_final.fit(X_train, y_train)
    yp_rf_tr, yp_rf_val, yp_rf_te = (rf_final.predict(X) for X in (X_train, X_val, X_test))
    print(f"RF test MAE={mean_absolute_error(y_test, yp_rf_te):.4f}  "
          f"R2={r2_score(y_test, yp_rf_te):.4f}")

    # ---- Stage 2: residuals + derivative (min-max to [0,1], paper Sec. 2.3.2) ----
    def resid_feats(y, yp):
        r = y - yp
        return r, np.diff(r, prepend=r[0])

    r_tr, d_tr = resid_feats(y_train, yp_rf_tr)
    r_va, d_va = resid_feats(y_val,   yp_rf_val)
    r_te, d_te = resid_feats(y_test,  yp_rf_te)

    rs, ds = MinMaxScaler((0, 1)), MinMaxScaler((0, 1))
    z_tr = np.hstack([rs.fit_transform(r_tr[:, None]), ds.fit_transform(d_tr[:, None])])
    z_va = np.hstack([rs.transform(r_va[:, None]),     ds.transform(d_va[:, None])])
    z_te = np.hstack([rs.transform(r_te[:, None]),     ds.transform(d_te[:, None])])

    Xb_tr, yb_tr = make_sequences(z_tr, BCO_SEQ_LEN)
    Xb_va, yb_va = make_sequences(z_va, BCO_SEQ_LEN)
    Xf_tr, yf_tr = make_sequences(z_tr, FINAL_SEQ_LEN)
    Xf_va, yf_va = make_sequences(z_va, FINAL_SEQ_LEN)
    lstm_sub = np.random.default_rng(SEED + 7).choice(len(Xb_tr), min(LSTM_SUBSET, len(Xb_tr)), replace=False)

    def bilstm_fitness(pos):
        tf.random.set_seed(SEED)
        p = decode_bilstm(pos)
        try:
            m = make_bilstm(p, Xb_tr.shape[1:], use_recurrent_dropout=False)
            es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
            m.fit(Xb_tr[lstm_sub], yb_tr[lstm_sub], batch_size=p['batch_size'],
                  epochs=p['epochs'], validation_data=(Xb_va, yb_va),
                  callbacks=[es], verbose=0)
            mse = float(m.evaluate(Xb_va, yb_va, verbose=0)[0])
            del m; K.clear_session(); gc.collect()
            return mse, None
        except Exception:
            traceback.print_exc(); K.clear_session(); gc.collect()
            return 1e6, None

    # ---- Stage 3: IBCO -> BiLSTM (compute is heavy -> smaller budget) ----
    bi_opt = IBCO(bilstm_fitness, BI_LOW, BI_UP, pop_size=10, max_iter=30,
                  eval_female=False, name="BiLSTM (9 params)",
                  checkpoint='bilstm_ibco_checkpoint.json')
    bi_x, bi_f, _ = bi_opt.optimize()
    bi_params = decode_bilstm(bi_x)
    print("\nBest BiLSTM hyper-parameters:")
    for k, v in bi_params.items():
        print(f"  {k:>20}: {v}")

    # ---- Stage 4: final BiLSTM ----
    final = make_bilstm(bi_params, Xf_tr.shape[1:], use_recurrent_dropout=True)
    final.fit(Xf_tr, yf_tr, batch_size=bi_params['batch_size'],
              epochs=min(bi_params['epochs'] * 3, 500),
              validation_data=(Xf_va, yf_va),
              callbacks=[EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
                         ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)],
              verbose=1)
    final.save('final_ibco_bilstm.keras')

    # ---- Stage 5: test evaluation  y_hat = y_RF + eps_BiLSTM  (Eq. 8) ----
    def predict_resid(z, L):
        Xs, _ = make_sequences(z, L)
        p = final.predict(Xs, verbose=0).flatten()
        return rs.inverse_transform(p[:, None]).flatten()

    pr_te = predict_resid(z_te, FINAL_SEQ_LEN)
    n_out = len(pr_te)
    yt, yrf = y_test[-n_out:], yp_rf_te[-n_out:]
    yhat = yrf + pr_te


# ======================================================
## Part-VII Plot all necessary Figures
# ======================================================

    print("\n" + "=" * 70 + "\nTEST RESULTS\n" + "=" * 70)
    print(f"{'Metric':<8}{'RF only':>12}{'IBCO RF+BiLSTM':>18}")
    for nm, a, b in [("MAE", mean_absolute_error(yt, yrf), mean_absolute_error(yt, yhat)),
                     ("RMSE", np.sqrt(mean_squared_error(yt, yrf)), np.sqrt(mean_squared_error(yt, yhat))),
                     ("MASE", mase(yt, yrf, y_train), mase(yt, yhat, y_train)),
                     ("R2", r2_score(yt, yrf), r2_score(yt, yhat))]:
        print(f"{nm:<8}{a:>12.4f}{b:>18.4f}")

    # ---- Stage 6: probabilistic metrics (validation residuals define the intervals) ----
    pr_va = predict_resid(z_va, FINAL_SEQ_LEN)
    n_v = len(pr_va)
    val_res = y_val[-n_v:] - (yp_rf_val[-n_v:] + pr_va)
    print("\nPrediction intervals (test):")
    print(f"{'Level':>6}{'PICP%':>9}{'PINAW':>9}{'AWD':>9}{'ACE%':>8}")
    for lv in (0.95, 0.90, 0.80, 0.70, 0.60):
        m = interval_metrics(yt, yhat, val_res, lv)
        print(f"{lv*100:>5.0f}%{m['PICP']:>9.2f}{m['PINAW']:>9.4f}{m['AWD']:>9.4f}{m['ACE']:>8.2f}")

    # ---- Plots ----
    n_plot = min(500, n_out)
    fig, ax = plt.subplots(3, 1, figsize=(13, 12))
    ax[0].plot(yt[:n_plot], label='Actual'); ax[0].plot(yrf[:n_plot], label='RF', alpha=.8)
    ax[0].plot(yhat[:n_plot], label='IBCO RF+BiLSTM', alpha=.8)
    ax[0].set_title('Predictions vs actual'); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(rf_opt.history, label='RF'); ax[1].plot(bi_opt.history, label='BiLSTM')
    ax[1].set_title('IBCO convergence (validation MSE)'); ax[1].set_yscale('log')
    ax[1].legend(); ax[1].grid(alpha=.3)
    ax[2].plot(rf_opt.div_history, label='RF'); ax[2].plot(bi_opt.div_history, label='BiLSTM')
    ax[2].axhline(0.05, ls='--', c='k', lw=.8); ax[2].set_title('Normalised diversity (Eq. 43)')
    ax[2].legend(); ax[2].grid(alpha=.3)
    plt.tight_layout(); plt.savefig('results_ibco.png', dpi=150)
    print("\nDone -> results_ibco.png")